In [1]:
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q groq
!pip install -q tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
import os
from tqdm import tqdm

In [3]:
GROQ_API_KEY = "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"
Client = Groq(api_key=GROQ_API_KEY)

In [4]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
reader = PdfReader("/content/Union_Budget_Analysis-2026-27.pdf")

text = ""

for page in reader.pages:
  text += page.extract_text()

In [7]:
chunk_size = 500
overlap = 100

chunks = []
start = 0

while start < len(text):
  end = start +chunk_size
  chunks.append(text[start:end])
  start += chunk_size-overlap
print("Total Chunks:", len(chunks))

Total Chunks: 92


In [8]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(name="unionbudget10101")

In [10]:
for i, chunk in enumerate(tqdm(chunks)):
  embedding = embedding_model.encode(chunk).tolist()
  collection.add(
      ids=str(i),
      documents=[chunk],
      embeddings=[embedding]
  )

100%|██████████| 92/92 [00:08<00:00, 11.33it/s]


In [11]:
def retrieve(query, n_results=3):
  query_embedding = embedding_model.encode(query).tolist()
  results = collection.query(
      query_embeddings=[query_embedding],
      n_results=n_results
  )
  return results["documents"][0]

In [19]:
def ask_llm(question):
  response = Client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role":"user",
              "content":question
          }
      ]
  )
  return response.choices[0].message.content

In [22]:
def ask_rag(question):
  docs = retrieve(question)
  context = "\n\n".join(docs)

  prompt = f"""
  You will be answering only from the provided context and not think anything of your own. if the answer is not present, return by saying "I could not find the information inside the document."
  Context: {context}
  Question: {question}
  """
  response = Client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[
          {
              "role":"user",
              "content":prompt
          }
      ]
  )
  return response.choices[0].message.content

In [14]:
query = "What is the allocation for the Defence Sector?"

docs = retrieve(query)
for i,d in enumerate(docs):
  print("="*80)
  print("Chunk", i+1)
  print(d)

Chunk 1
 Defence has the highest allocation in 2026-27, at Rs 7,84,678 crore, accounting for 15% of the total 
budgeted expenditure of the central government.  Other ministries with high allocation s include: (i) Road Transport and 
Highways (6% of total expenditure), (ii) Railways (5%), and (iii) Home Affairs (5%).   
Table 4: Ministry-wise expenditure in 2026-27 (Rs crore) 
  Actuals 
2024-25 
Budgeted 
2025-26 
Revised 
2025-26 
Budgeted 
2026-27 
% change (2025-26 RE 
 to 2026-27 BE) 
Defence 6,36,0
Chunk 2
03 41,437 94,808 128.8% 
Housing and Urban Affairs 53,255 96,777 57,204 85,522 49.5% 
Total Expenditure 46,52,867 50,65,345 49,64,842 53,47,315 7.7% 
Sources: Expenditure Budget, Union Budget 2026-27; PRS. 
 
* Corrected on February 2, 2026.  
Union Budget 2026-27 Analysis  PRS Legislative Research 
 
February 1, 2026  - 6 - 
 
▪ Ministry of Defence: Allocation is estimated to increase by Rs 52,166 crore (7%) in 2026-27, over the revised estimate 
of 2025-26.  The allocation tow

In [17]:
question = "What is the allocation for the Defence Sector?"

In [20]:
print(ask_llm(question))

I'm not able to provide the current allocation for the Defence Sector as my knowledge cutoff is 01 March 2023, and I do not have access to real-time information. The allocation for the Defence Sector can vary depending on the country and the specific budget being referred to.

If you're looking for the most up-to-date information, I recommend checking the latest budget documents or official government websites for the most accurate and current information. Additionally, you can also try searching for news articles or press releases from reputable sources that may have reported on the Defence Sector allocation.


In [23]:
print(ask_rag(question))

The allocation for the Defence Sector in 2026-27 is Rs 7,84,678 crore, which accounts for 15% of the total budgeted expenditure of the central government.
